# Model Comparison: LRT, AIC, and Non-Nested Tests

## Overview

Model comparison asks: given the data, which model is preferred? The right method depends on the model relationship and inferential goal.

**Comparison methods:**

| Method | Models | Goal | Based on |
|---|---|---|---|
| **LRT** | Nested, same family | Hypothesis test | χ² distribution |
| **F-test** | Nested OLS | Hypothesis test | F distribution |
| **AIC / AICc** | Any models, same data | Predictive preference | K-L divergence |
| **BIC** | Any models, same data | Parsimony / consistency | Bayesian marginal likelihood |
| **Vuong test** | Non-nested | Hypothesis test | LR statistic |
| **Likelihood ratio (mixed)** | Nested mixed models | Hypothesis test | Parametric bootstrap |

**Nested vs. non-nested:** Model A is nested in Model B if A can be obtained by fixing some of B's parameters to zero or a constant. Poisson is nested in NB (θ→∞); a model with one predictor is nested in a model with two.

**Critical rule:** Models compared by likelihood must be fitted to the **identical** dataset — same observations, same response, same offset. Any difference in n invalidates the comparison.

---

## Setup

In [1]:
library(tidyverse)
library(ggplot2)
library(lme4)          # mixed models
library(lmtest)        # lrtest(), waldtest()
library(AICcmodavg)    # AICc()
library(pscl)          # vuong()
library(MASS)          # glm.nb()
library(pbkrtest)      # parametric bootstrap LRT for mixed models
library(patchwork)

set.seed(42)

n <- 200
comp_data <- tibble(
  catchment  = sample(1:20, n, replace=TRUE),
  nitrate    = runif(n, 1, 10),
  water_qual = runif(n, 2, 9),
  elevation  = rnorm(n, 200, 80),
  mu         = exp(1.8 - 0.25*nitrate + 0.18*water_qual),
  count      = rnbinom(n, mu=mu, size=2)
)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'lme4' was built under R version 4.4.3"
Loading required package: Matrix


Attaching package: 'Matrix'


Th

---

## Likelihood Ratio Test for Nested Models

In [2]:
# LRT: 2*(logLik(full) - logLik(reduced)) ~ χ²(df = difference in parameters)

m0 <- glm(count ~ 1,                           data=comp_data, family=poisson)
m1 <- glm(count ~ nitrate,                     data=comp_data, family=poisson)
m2 <- glm(count ~ nitrate + water_qual,        data=comp_data, family=poisson)
m3 <- glm(count ~ nitrate + water_qual + elevation, data=comp_data, family=poisson)

# Sequential LRT
lrt_seq <- lmtest::lrtest(m0, m1, m2, m3)
print(lrt_seq)

# Specific comparison: does elevation improve on m2?
lrt_elev <- lmtest::lrtest(m2, m3)
cat(sprintf("\nLRT (elevation added): χ²=%.3f, df=%d, p=%.4f\n",
            lrt_elev$Chisq[2], lrt_elev$Df[2], lrt_elev$`Pr(>Chisq)`[2]))
cat(ifelse(lrt_elev$`Pr(>Chisq)`[2] < 0.05,
           "→ elevation significantly improves model fit\n",
           "→ elevation does not significantly improve model fit\n"))

Likelihood ratio test

Model 1: count ~ 1
Model 2: count ~ nitrate
Model 3: count ~ nitrate + water_qual
Model 4: count ~ nitrate + water_qual + elevation
  #Df  LogLik Df    Chisq Pr(>Chisq)    
1   1 -823.81                           
2   2 -686.04  1 275.5398    < 2e-16 ***
3   3 -611.09  1 149.8831    < 2e-16 ***
4   4 -607.75  1   6.6985    0.00965 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

LRT (elevation added): χ²=6.698, df=1, p=0.0096
→ elevation significantly improves model fit


---

## AIC/AICc Comparison

In [3]:
# AIC comparison: works for nested AND non-nested models
# Does not require same number of parameters; does require same data
m_nb  <- MASS::glm.nb(count ~ nitrate + water_qual, data=comp_data)

model_list <- list(
  null      = m0,
  nitrate   = m1,
  nit_wq    = m2,
  nit_wq_el = m3,
  neg_bin   = m_nb
)

aic_tab <- map_dfr(names(model_list), function(nm) {
  m <- model_list[[nm]]
  tibble(
    model  = nm,
    family = family(m)$family,
    k      = length(coef(m)),
    AIC    = round(AIC(m), 2),
    AICc   = round(AICcmodavg::AICc(m), 2),
    BIC    = round(BIC(m), 2)
  )
}) %>%
  mutate(
    delta_AICc = round(AICc - min(AICc), 2),
    weight     = round(exp(-0.5*delta_AICc) / sum(exp(-0.5*delta_AICc)), 4)
  ) %>%
  arrange(AICc)

print(aic_tab)
# NB model should rank first: true DGP is NB
# BIC typically selects simpler model than AICc

# A tibble: 5 × 8
  model     family                        k   AIC  AICc   BIC delta_AICc weight
  <chr>     <chr>                     <int> <dbl> <dbl> <dbl>      <dbl>  <dbl>
1 neg_bin   Negative Binomial(2.1375)     3 1014. 1014. 1027.         0       1
2 nit_wq_el poisson                       4 1223. 1224. 1237.       209.      0
3 nit_wq    poisson                       3 1228. 1228. 1238.       214.      0
4 nitrate   poisson                       2 1376. 1376. 1383.       362.      0
5 null      poisson                       1 1650. 1650. 1653.       635.      0


---

## Comparing Mixed Models: Fixed and Random Effects

In [4]:
# ── Comparing fixed effects: use ML (not REML) ───────────────────────────────
# REML estimates are not comparable across models with different fixed effects
lmm_null <- lme4::lmer(count ~ 1 + (1|catchment),
                        data=comp_data, REML=FALSE)
lmm_nit  <- lme4::lmer(count ~ nitrate + (1|catchment),
                        data=comp_data, REML=FALSE)
lmm_full <- lme4::lmer(count ~ nitrate + water_qual + (1|catchment),
                        data=comp_data, REML=FALSE)

cat("LRT: fixed effects (ML):\n")
print(anova(lmm_null, lmm_nit, lmm_full))

# ── Comparing random effects: use REML ───────────────────────────────────────
# Must have IDENTICAL fixed effects structure
lmm_re1 <- lme4::lmer(count ~ nitrate + water_qual + (1|catchment),
                        data=comp_data, REML=TRUE)
lmm_re2 <- lme4::lmer(count ~ nitrate + water_qual + (nitrate|catchment),
                        data=comp_data, REML=TRUE)

cat("\nLRT: random effects structure (REML, parametric bootstrap):\n")
# Standard LRT is anticonservative for random effects; use parametric bootstrap
pb_test <- pbkrtest::PBmodcomp(
  lmm_re2, lmm_re1,
  nsim  = 200,   # increase to 1000 for publication
  seed  = 42
)
summary(pb_test)

boundary (singular) fit: see help('isSingular')



LRT: fixed effects (ML):
Data: comp_data
Models:
lmm_null: count ~ 1 + (1 | catchment)
lmm_nit: count ~ nitrate + (1 | catchment)
lmm_full: count ~ nitrate + water_qual + (1 | catchment)
         npar    AIC    BIC  logLik -2*log(L)  Chisq Df Pr(>Chisq)    
lmm_null    3 1279.7 1289.6 -636.84    1273.7                         
lmm_nit     4 1236.6 1249.8 -614.31    1228.6 45.061  1  1.910e-11 ***
lmm_full    5 1206.1 1222.6 -598.06    1196.1 32.495  1  1.195e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


boundary (singular) fit: see help('isSingular')




LRT: random effects structure (REML, parametric bootstrap):


boundary (singular) fit: see help('isSingular')



Bootstrap test; time: 6.84 sec; samples: 200; extremes: 5;
Requested samples: 200 Used samples: 164 Extremes: 5
large : count ~ nitrate + water_qual + (nitrate | catchment)
           stat     df   ddf  p.value   
LRT      4.7775 2.0000       0.091745 . 
PBtest   4.7775              0.036364 * 
Gamma    4.7775              0.038504 * 
Bartlett 9.2952 2.0000       0.009584 **
F        2.3887 2.0000 73.57 0.098827 . 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


---

## Common Pitfalls

**1. Comparing models fitted to different datasets**  
LRT and AIC are only valid when computed on identical data. If models differ in which observations were dropped due to missing values in different predictors, the log-likelihoods are not comparable. Always fit all candidate models to the same complete-case dataset, or use multiple imputation consistently across all models.

**2. Using LRT to compare non-nested models**  
LRT requires one model to be a special case of the other. Comparing a Poisson to a log-normal model is not a nested comparison — the distributional families are different. Use AIC for non-nested comparisons; use the Vuong test when you need a formal hypothesis test between two non-nested models.

**3. Using standard LRT for random effects testing**  
The standard LRT for testing whether a variance component is zero has a null distribution that is a mixture of chi-squared distributions, not a simple χ²(1). The standard p-value is anticonservative (too small). Use the parametric bootstrap (`pbkrtest::PBmodcomp()`) or halve the p-value from the standard LRT as a rough correction.

**4. Comparing REML fits with different fixed effects**  
REML likelihood is computed after integrating out fixed effects — different fixed effect structures produce non-comparable REML likelihoods. Always refit with `REML=FALSE` before comparing fixed effects structures, and switch back to `REML=TRUE` for the final model.

**5. Treating a non-significant LRT as evidence that the simpler model is correct**  
A non-significant LRT means the data are consistent with the simpler model — it does not confirm that the simpler model is true. The test may have low power with small samples. Report confidence intervals for the additional parameter(s) alongside the LRT result.

---
*r_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*